# 2.3.2 Native Sparse Attention(NSA)

motivation: 
- reduce attention computation 减少attention计算量 
- avoid perfomance degration - native / trainable 避免性能下降 - 原生/可训练

## Design
- **compressed attention**: holding global information/context
- **selection attention**: holding local information/context
- **sliding window attention**: holding related information/context

## Example

![architecture](../../figs/2.3.2-NSA-architecture.png)

*This picture is the overall architecture of NSA. (From NSA paper, see **Reference 1**)*

We have now calculated context length of 32 tokens. So we have the corresponding 32 KV cache. 
**Target**: caculate the new and less KV cache to compute with new query.

First, divide the 32 into 4 blocks, the block is of $len_b=8$

- **compression**: For each block, compress it into $len=len_b / c, c \le len_b$.
- **selection**: Select $s$ block from all the blocks.
- **sliding**: Select the former $l$ KV in the whole sequence. (e.g select 24-32 KV cache)

Last, use **gate** to gather all computed KV. 
$$KV_{new} = g^{cmp} o^{cmp} + g^{sle} o^{sle} + g^{win} o^{win}$$

The parameter $g$ is calculated from the linear layer output with the input $x_{32}$, which can be learned during training.

## Attention Implementation

In [3]:
import torch

batch_size = 32
t = 32 # input sequence length
dim = 64 # hidden dimension

X = torch.randn(batch_size, t, dim)

W_q = torch.randn(dim, dim)
W_k = torch.randn(dim, dim)
W_v = torch.randn(dim, dim)

# batch_size x t x dim
Q = X @ W_q
K = X @ W_k
V = X @ W_v

The blocks are not simply divided by spliting KV. 

> Why we need stride?

We made the block overlap for better accuracy.

In [ ]:
block_size = 8
stride = 4
num_blocks = (t - block_size) // stride + 1

### Compressed Attention

[single head version]

In [ ]:
W_k_cmp = torch.randn(block_size, 1) # multiple block using one weight?
W_v_cmp = torch.randn(block_size, 1)
W_pe = torch.randn(block_size, dim) # positional encoding

K_cmp = [] 
V_cmp = []
for i in range(num_blocks):
    cur_K = K[:, i*stride:i*stride+block_size, :] + W_pe.unsqueeze(0) # batch_size x block_size x dim
    cur_V = V[:, i*stride:i*stride+block_size, :] + W_pe.unsqueeze(0)
    cur_K_cmp = cur_K.transpose(1,2) @ W_k_cmp # batch_size x dim x 1
    cur_V_cmp = cur_V.transpose(1,2) @ W_v_cmp # batch_size x dim x 1
    K_cmp.append(cur_K_cmp) # (batch_size x dim x 1) x num_blocks
    V_cmp.append(cur_V_cmp)

K_cmp = torch.cat(K_cmp, dim=2).transpose(1, 2) # batch_size x num_blocks x dim
V_cmp = torch.cat(V_cmp, dim=2).transpose(1, 2) # batch_size x num_blocks x dim

[multi-head version]

In [ ]:
num_heads = 8
head_dim = dim // num_heads

Q_mha = Q.view(batch_size, t, num_heads, head_dim).transpose(1, 2) # batch_size x num_heads x t x head_dim
K_cmp_mha = K_cmp.view(batch_size, num_blocks, num_heads, head_dim).transpose(1, 2) # batch_size x num_heads x num_blocks x head_dim
V_cmp_mha = V_cmp.view(batch_size, num_blocks, num_heads, head_dim).transpose(1, 2) # batch_size x num_heads x num_blocks x head_dim

score_cmp = Q_mha @ K_cmp_mha.transpose(2,3) / (head_dim ** 0.5) # batch_size x num_heads x t x num_blocks
p_cmp = torch.softmax(score_cmp, dim=-1) # batch_size x num_heads x t x num_blocks
o_cmp = p_cmp @ V_cmp_mha # batch_size x num_heads x t x head_dim

### Selection Attention

Select the top-k highest attention score KV-cache. Note that for different KV head, we select different part. But for GQA and so on, we select the same kv despite different queries in one head for faster speed.

Here's what the original paper writes:
> For models employing GQA or MQA where key-value caches are shared across query heads, consistent block selection across these heads has to be ensured to minimize KV cache loading during decoding. The shared importance scores across heads in a group are formally defined as.

In [ ]:
select_top_k = 2

p_slc = p_cmp.sum(dim=1) # batch_size x t x num_blocks
_, idx = torch.topk(p_slc, select_top_k, dim=2) # batch_size x t x select_top_k

# select top-k whole blocks from the original KV
K_slc = torch.randn(batch_size, t, select_top_k*block_size, dim) # every query token has its own selected KV blocks
V_slc = torch.randn(batch_size, t, select_top_k*block_size, dim)

for i in range(batch_size):
    for j in range(t):
        for k in range(select_top_k):
            block_idx = idx[i, j, k].item()
            K_slc[i, j, k*stride:k*stride+block_size, :] = K[i, block_idx*stride:block_idx*stride+block_size, :]
            V_slc[i, j, k*stride:k*stride+block_size, :] = V[i, block_idx*stride:block_idx*stride+block_size, :]

In [ ]:
# selection attention calculation
# divide K_slc into heads
K_slc_mha = K_slc.view(batch_size, t, select_top_k*block_size, num_heads, head_dim).transpose(2, 3) # batch_size x t x num_heads x select_top_k*block_size x head_dim
K_slc = K_slc_mha.sum(dim = 2, keepdim=True) # batch_size x t x 1 x select_top_k*block_size x head_dim

V_slc_mha = V_slc.view(batch_size, t, select_top_k*block_size, num_heads, head_dim).transpose(2, 3) # batch_size x t x num_heads x select_top_k*block_size x head_dim
V_slc = V_slc_mha.sum(dim = 2, keepdim=True) # batch_size x t x 1 x select_top_k*block_size x head_dim

o_slc = torch.zeros(batch_size, t, dim) # batch_size x t x dim
for i in range(t):
    Q_slc_i = Q_mha[:, :, i, :].unsqueeze(2) # batch_size x num_heads x 1 x head_dim
    K_slc_i = K_slc[:, i, :, :, :].repeat(1, num_heads, 1, 1) # batch_size x num_heads x select_top_k*block_size x head_dim
    V_slc_i = V_slc[:, i, :, :, :].repeat(1, num_heads, 1, 1) # batch_size x num_heads x select_top_k*block_size x head_dim

    attn_score_i = Q_slc_i @ K_slc_i.transpose(2, 3) / (head_dim ** 0.5) # batch_size x num_heads x 1 x select_top_k*block_size
    p_slc_i = torch.softmax(attn_score_i, dim=-1) # batch_size x num_heads x 1 x select_top_k*block_size

    o_slc_i = p_slc_i @ V_slc_i # batch_size x num_heads x 1 x head_dim
    o_slc[:, i, :] = o_slc_i.transpose(1, 2).reshape(batch_size, 1, dim)

### Sliding Window Attention

Select the nearest KV cache through mask.

In [1]:
def get_window_mask(t, window_size):
    mask = torch.ones(t, t)
    mask = torch.tril(mask)
    sub = torch.tril(-torch.ones(t,t), diagonal=-window_size)
    mask = mask + sub
    return mask

In [4]:
get_window_mask(7,3)

tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0.],
        [0., 1., 1., 1., 0., 0., 0.],
        [0., 0., 1., 1., 1., 0., 0.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 0., 1., 1., 1.]])

In [ ]:
window_size = 4

score = Q @ K.transpose(1, 2) / (dim ** 0.5) # batch_size x t x t
score = torch.softmax(score, dim=-1)
score = score * get_window_mask(t, window_size)

o_win = score @ V # batch_size x t x dim

### Gate

Gate is a MLP+softmax layer.

$$o = \sum_{c\in C} g_t^c \cdot Attn(q_t, K_t^c, V_t^c), g_t^c \in [0,1]$$

In [ ]:
W_gate = torch.randn(dim, 3)
gate = torch.sigmoid(X @ W_gate) # batch_size x t x 3

o_list = [o_cmp, o_slc, o_win]
o = sum(gate[:,:,i].unsqueeze(2) * o_list[i] for i in range(3)) # batch_size x t x dim

## Optimization

Compressed Attention & Sliding Window Attention are compatible with FlashAttention because they have nothing to do with kv block selection.

## SGLang Implementation
?

## references

1. (paper) Native Sparse Attention: Hardware-Aligned and Natively Trainable Sparse Attention, https://arxiv.org/pdf/2502.11089
2. (blog) 【手撕NSA】DeepSeek新作-原生稀疏注意力-超长文(附代码), https://zhuanlan.zhihu.com/p/24841366485